# CAIA / PROFILE summary-stats figures

Thin driver around `summary_figures.py`. Run the cells top-to-bottom to:
1. Load both cohorts (with timing fields recomputed from datetimes for CAIA).
2. Emit overlaid figures for record-span, dx→tx, time-to-platinum, platinum incidence + KM, overall-survival KM, and per-lab raw + IRNT histograms.
3. Inspect a handful of the produced PNGs inline.

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path('/data/gusev/USERS/jpconnor/code/CAIA')
CAIA_DIR     = PROJECT_ROOT / 'COMPASS' / 'survival_analysis' / 'CAIA'

NEPC_PROJ_PATH = Path('/data/gusev/USERS/jpconnor/data/CAIA/COMPASS')
PROFILE_CSV    = NEPC_PROJ_PATH / 'longitudinal_prediction_data.csv'
CAIA_PARQUET   = NEPC_PROJ_PATH / 'caia_compass_longitudinal.parquet'

OUTPUT_DIR     = Path('/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS/summary_stats')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)
for _p in (str(CAIA_DIR.parent), str(CAIA_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print('profile_csv  ', PROFILE_CSV)
print('caia_parquet ', CAIA_PARQUET)
print('output_dir   ', OUTPUT_DIR)

In [ ]:
import summary_figures as sf

sf.run(
    profile_csv=PROFILE_CSV,
    caia_parquet=CAIA_PARQUET,
    output_dir=OUTPUT_DIR,
    formats=('png', 'pdf'),
    min_coverage=0.20,
)

In [ ]:
from IPython.display import Image, display
for name in ['time_distributions.png', 'platinum_incidence.png', 'overall_survival_km.png']:
    p = OUTPUT_DIR / name
    if p.exists():
        print(name)
        display(Image(str(p)))
    else:
        print(f'  missing: {p}')

In [ ]:
# Spot-check a handful of per-lab panels (raw + IRNT, side-by-side).
# Picks up to 6 labs that summary_figures.py actually wrote, so the cell
# adapts to whatever the cohort union produced (LOINC -> short name folding
# means file stems are the canonical short names from helpers.plotting).
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

raw_dir  = OUTPUT_DIR / 'labs_raw'
irnt_dir = OUTPUT_DIR / 'labs_irnt'

raw_stems  = {p.stem for p in raw_dir.glob('*.png')}
irnt_stems = {p.stem for p in irnt_dir.glob('*.png')}
both       = sorted(raw_stems & irnt_stems)
raw_only   = sorted(raw_stems - irnt_stems)
irnt_only  = sorted(irnt_stems - raw_stems)

print(f'panels written: raw={len(raw_stems)}  irnt={len(irnt_stems)}  both={len(both)}')
if raw_only:  print(f'raw-only:  {raw_only}')
if irnt_only: print(f'irnt-only: {irnt_only}')

# Prefer labs that have both panels; fall back to whatever exists.
picks = both[:6] if both else sorted(raw_stems | irnt_stems)[:6]
if not picks:
    print('No per-lab panels written — did summary_figures.run() complete?')
else:
    for stem in picks:
        raw_p  = raw_dir  / f'{stem}.png'
        irnt_p = irnt_dir / f'{stem}.png'
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        for ax, p in zip(axes, [raw_p, irnt_p]):
            ax.axis('off')
            if p.exists():
                ax.imshow(mpimg.imread(p))
        plt.suptitle(stem.replace('_', ' '))
        plt.tight_layout()
        plt.show()